# SOCCAT — Pipeline Demo

A short end-to-end demonstration of the SOCCAT pipeline on five example sentences.

The pipeline has two steps:

1. **Step 1 — Social-group mention detection.** A binary classifier predicts whether a sentence mentions any social group at all.
2. **Step 2 — Category classification.** For sentences flagged in step 1, fine-tuned NLI models predict which specific social groups are mentioned. The full SOCCAT taxonomy has eight categories (each with its own classifier); this demo runs three of them for brevity.

All models are loaded from the Hugging Face Hub at https://huggingface.co/selsar. The first run downloads them (a few hundred MB per model); subsequent runs use the local cache.


## 1. Setup


In [1]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification

pd.set_option("display.max_colwidth", 80)

## 2. Example sentences

A small multilingual sample (German and French) covering a mix of sentences that do and do not mention social groups.


In [2]:
sentences = [
    "Die Arbeitslosen demonstrierten gestern vor dem Bundestag.",
    "Les retraités s'inquiètent de la nouvelle réforme des pensions.",
    "Heute scheint die Sonne in Berlin.",
    "Muslime und Christen feierten gemeinsam in der Innenstadt.",
    "Les agriculteurs bloquent les autoroutes pour protester.",
]

pd.DataFrame({"sentence": sentences})

,sentence
0,Die Arbeitslosen demonstrierten gestern vor dem Bundestag.
1,Les retraités s'inquiètent de la nouvelle réforme des pensions.
2,Heute scheint die Sonne in Berlin.
3,Muslime und Christen feierten gemeinsam in der Innenstadt.
4,Les agriculteurs bloquent les autoroutes pour protester.


## 3. Step 1 — Binary mention detection

Load `selsar/social_group_detection` and predict whether each sentence mentions any social group.
Label convention: `0` = no group mentioned, `1` = group mentioned.


In [3]:
STEP1_MODEL = "selsar/social_group_detection"

tokenizer_1 = AutoTokenizer.from_pretrained(STEP1_MODEL)
model_1 = AutoModelForSequenceClassification.from_pretrained(STEP1_MODEL)
model_1.eval()

enc = tokenizer_1(sentences, padding=True, truncation=True, return_tensors="pt")
with torch.no_grad():
    logits = model_1(**enc).logits
probs = torch.softmax(logits, dim=-1)
mention_pred = logits.argmax(dim=-1).tolist()
mention_prob = probs[:, 1].tolist()

step1_df = pd.DataFrame({
    "sentence": sentences,
    "mentions_group": [bool(p) for p in mention_pred],
    "prob_mention": [round(p, 3) for p in mention_prob],
})
step1_df

ModuleNotFoundError: Could not import module 'DebertaV2ForSequenceClassification'. Are this object's requirements defined correctly?

## 4. Step 2 — Category classification

For each sentence flagged in step 1, run the per-category NLI classifiers. The hypothesis follows the template used in training:

> `This sentence refers to <category> as a social group, specifically "<label>".`

An entailment probability above 0.5 is treated as a positive prediction for that (sentence, label) pair.

To keep the demo quick, three of the eight categories are run here. To see the full pipeline, add the remaining five (`socio_economic_position`, `age_and_family_status`, `social_roles_and_behavior`, `social_deviance`, `real_estate_ownership`) — the labels and hub repo names are in `src/step_2/categories.json`.


In [ ]:
DEMO_CATEGORIES = [
    {
        "name": "labor_market_position",
        "display": "labor market position",
        "hub_repo": "selsar/cv_labor_market_position",
        "labels": [
            "wage and salary earners", "civil servants", "CEOs and corporate leaders",
            "employers", "entrepreneurs", "self-employed and freelancers",
            "unemployed", "retirees", "housewives and househusbands",
        ],
    },
    {
        "name": "identities",
        "display": "identities and minority/majority status",
        "hub_repo": "selsar/cv_identities",
        "labels": [
            "men", "women", "cisgender and heterosexuals", "LGBTQIA+",
            "disabled people",
            "people with an immigration background, including immigrants",
            "Ethnic and racial minorities",
            "Christians", "Jews", "Muslims",
            "multiple (or other) religious or minority groups",
        ],
    },
    {
        "name": "profession",
        "display": "profession",
        "hub_repo": "selsar/cv_profession",
        "labels": [
            "athletes", "authors and artists", "doctors",
            "farmers and fishermen", "health and care professionals",
            "journalists", "legal professionals",
            "politicians and high-ranking officials",
            "sex workers", "scientists and professors",
            "security forces", "soldiers", "teachers and educators",
            "other professions",
        ],
    },
]


def make_hypothesis(category_display, label):
    return (
        f'This sentence refers to {category_display} as a social group, '
        f'specifically "{label}".'
    )

In [ ]:
positive_sentences = [s for s, p in zip(sentences, mention_pred) if p == 1]
print(f"Step 1 flagged {len(positive_sentences)} of {len(sentences)} sentences as mentioning a group.\n")

predicted_groups = {s: [] for s in positive_sentences}

for cat in DEMO_CATEGORIES:
    print(f"Loading {cat['hub_repo']} ...")
    tokenizer_2 = AutoTokenizer.from_pretrained(cat["hub_repo"])
    model_2 = AutoModelForSequenceClassification.from_pretrained(cat["hub_repo"])
    model_2.eval()

    for sent in positive_sentences:
        premises = [sent] * len(cat["labels"])
        hypotheses = [make_hypothesis(cat["display"], lbl) for lbl in cat["labels"]]
        enc = tokenizer_2(
            premises, hypotheses,
            padding=True, truncation=True, max_length=256,
            return_tensors="pt",
        )
        with torch.no_grad():
            logits = model_2(**enc).logits
        # label convention: 0 = entailment (positive), 1 = not_entailment
        prob_entail = torch.softmax(logits, dim=-1)[:, 0]
        for label, p in zip(cat["labels"], prob_entail.tolist()):
            if p > 0.5:
                predicted_groups[sent].append({
                    "category": cat["display"],
                    "label": label,
                    "prob_entail": round(p, 3),
                })

print("\nDone.")

## 5. Final pipeline output

Combined view: one row per (sentence, predicted label) pair. Sentences with no detected groups appear once with `(none)` in the category and label columns.


In [ ]:
rows = []
for sent in sentences:
    if sent not in predicted_groups:
        rows.append({
            "sentence": sent, "mentions_group": False,
            "category": "(none)", "label": "(none)", "prob_entail": None,
        })
    elif not predicted_groups[sent]:
        rows.append({
            "sentence": sent, "mentions_group": True,
            "category": "(none predicted by demo categories)",
            "label": "(none predicted)", "prob_entail": None,
        })
    else:
        for g in predicted_groups[sent]:
            rows.append({
                "sentence": sent, "mentions_group": True,
                "category": g["category"], "label": g["label"],
                "prob_entail": g["prob_entail"],
            })

final_df = pd.DataFrame(rows)
final_df

---

**To extend this demo to the full pipeline:**

- Add the remaining five categories to `DEMO_CATEGORIES` (labels + hub repos in `src/step_2/categories.json`).
- Replace the example sentences with your own.
- For larger batches, use `replicate_from_hub_step_2.py` instead — it processes the per-category NLI CSV files in one pass with the HF `Trainer`.
